# 01 — Knowledge-base pipeline (documented walkthrough)

This notebook **is the documented orchestration** of the ICD-11 knowledge-base build
(the same flow as `components/main.py`).

## Pipeline overview

```
ICD-11 PDF  --chunker-->  icd11_chunks.json  --ingestion-->  ChromaDB vectors
```

| Step | Module | Input | Output |
|------|--------|-------|--------|
| 1. Chunking | `components/chunker.py` | `knowledge_base/icd_11.pdf` | `knowledge_base/icd11_chunks.json` |
| 2. Ingestion | `components/ingestion.py` | chunks JSON | `knowledge_base/chroma_db/` |

Heavy lifting stays in those modules (PDF parsing / embeddings). This notebook
holds the **control flow, explanation, and inspection** that used to live only
as a thin CLI in `main.py`.

CLI still works for automation: `python -m components.main`


In [ ]:
# Path setup — works from repo root or notebooks/
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "components" / "config.py").exists():
    root = _cwd
elif (_cwd.parent / "components" / "config.py").exists():
    root = _cwd.parent
else:
    raise RuntimeError("Open this notebook from the repo root or notebooks/ folder.")

sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "retriever"))
print("Project root:", root)


## Configuration

Paths and knobs come from `components/config.py` so every entrypoint agrees
on the same PDF page range, chunk path, and embedding model.


In [ ]:
from components.config import (
    PDF_PATH,
    CHUNKS_PATH,
    CHROMA_PATH,
    COLLECTION_NAME,
    EMBEDDING_MODEL,
    BATCH_SIZE,
    CONTENT_START_PAGE,
    CONTENT_END_PAGE,
)

print("PDF:          ", PDF_PATH)
print("  exists:     ", PDF_PATH.exists())
print("Chunks JSON:  ", CHUNKS_PATH)
print("  exists:     ", CHUNKS_PATH.exists())
print("ChromaDB:     ", CHROMA_PATH)
print("  exists:     ", CHROMA_PATH.exists())
print("Collection:   ", COLLECTION_NAME)
print("Embed model:  ", EMBEDDING_MODEL)
print("Batch size:   ", BATCH_SIZE)
print("PDF pages:    ", CONTENT_START_PAGE, "→", CONTENT_END_PAGE)


## Step 1 — Chunking (PDF → JSON)

### What this step does
1. Extract text from the ICD-11 CDDR PDF for the clinical page range.
2. Parse disorder codes / section headings into structured chunk dicts.
3. Split oversized sections with word overlap.
4. Write `knowledge_base/icd11_chunks.json`.

### Why it matters for RAG
Retrieval quality depends on chunk boundaries. We keep section-aware clinical
units (e.g. Essential Features, Boundary with Normality) rather than naive
fixed-size windows.

Set `RUN_CHUNKING = True` only when the PDF is present and you intend to rebuild.


In [ ]:
from components.chunker import (
    run_chunking,
    MAX_CHUNK_WORDS,
    CHUNK_WORD_OVERLAP,
)

RUN_CHUNKING = False  # True → regenerate JSON from PDF (slow; needs pdftotext + PDF)

print(f"MAX_CHUNK_WORDS={MAX_CHUNK_WORDS}, OVERLAP={CHUNK_WORD_OVERLAP}")

if RUN_CHUNKING:
    if not PDF_PATH.exists():
        raise FileNotFoundError(f"Missing PDF at {PDF_PATH}")
    print("\n=== Step 1/2: Chunking (same as components.main) ===")
    chunks = run_chunking(
        pdf_path=str(PDF_PATH),
        chunks_path=str(CHUNKS_PATH),
        start_page=CONTENT_START_PAGE,
        end_page=CONTENT_END_PAGE,
        max_words=MAX_CHUNK_WORDS,
        overlap_words=CHUNK_WORD_OVERLAP,
    )
    print(f"Wrote {len(chunks)} chunks → {CHUNKS_PATH}")
else:
    print("Skipping chunking (RUN_CHUNKING=False). Will inspect existing JSON.")


## Inspect chunks

After chunking (or if JSON already exists), inspect structure and content.
Each chunk typically includes disorder metadata plus clinical text fields used
for BM25 (`prompt_text`) and/or embedding (`embed_text` / `text`).


In [ ]:
import json
from collections import Counter

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        f"{CHUNKS_PATH} not found. Set RUN_CHUNKING=True or obtain the JSON artifact."
    )

with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
print("Fields:", sorted(chunks[0].keys()))
print("\nTop sections:")
for name, n in Counter(c.get("section", "<none>") for c in chunks).most_common(8):
    print(f"  {n:5d}  {name}")

print("\n--- Examples ---")
for c in chunks[:3]:
    body = (c.get("prompt_text") or c.get("text") or "").replace("\n", " ")
    print(f"\n[{c.get('disorder_code')}] {c.get('disorder_name')} — {c.get('section')}")
    print(body[:320], "...")


## Step 2 — Ingestion (JSON → ChromaDB)

### What this step does
1. Load chunk JSON.
2. Embed each chunk with BioLORD-2023 (`sentence-transformers`).
3. Upsert vectors into a persistent Chroma collection.

### Why BioLORD
Domain embeddings improve dense retrieval over general-purpose models for
clinical wording in ICD-11 text.

This step can take several minutes on CPU. Use `REBUILD_CHROMA=True` only when
you need a clean re-index.


In [ ]:
from components.ingestion import run_ingestion

RUN_INGESTION = False   # True → embed + upsert into Chroma
REBUILD_CHROMA = False  # True → delete existing collection first

if RUN_INGESTION:
    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(f"Missing chunks at {CHUNKS_PATH}")
    print("\n=== Step 2/2: Ingestion (same as components.main) ===")
    run_ingestion(
        chunks_path=str(CHUNKS_PATH),
        chroma_path=str(CHROMA_PATH),
        collection_name=COLLECTION_NAME,
        embedding_model_name=EMBEDDING_MODEL,
        batch_size=BATCH_SIZE,
        rebuild=REBUILD_CHROMA,
    )
    print("Ingestion finished.")
else:
    print("Skipping ingestion (RUN_INGESTION=False).")


## Full orchestration (equivalent to `components/main.py`)

The CLI `main()` is literally: optional chunking → ingestion → done.
The next cell mirrors that control flow so the thesis walkthrough stays in one place.


In [ ]:
# Mirrors components.main:main() — flip flags above, then run this cell.
SKIP_CHUNKING = not RUN_CHUNKING
# (Chunking / ingestion already executed in the cells above when flags are True.)

print("Orchestration summary")
print(f"  skip_chunking = {SKIP_CHUNKING}")
print(f"  ran_ingestion = {RUN_INGESTION}")
print(f"  rebuild       = {REBUILD_CHROMA}")
print("\nCLI equivalents:")
print("  python -m components.main")
print("  python -m components.main --skip-chunking")
print("  python -m components.main --rebuild")
print("\nNext: notebooks/02_dataset_prep_demo.ipynb")
